In [1]:
from gensim.models import Word2Vec, KeyedVectors
from tqdm import tqdm
import os
glove_path = 'glove.6B.50d.word2vec.txt'



In [2]:
TOKEN_DIR = '..\\Data\\preprocessed_tokens'

class TokenCorpus:
    def __init__(self, token_dir):
        self.files = [os.path.join(token_dir, f) for f in os.listdir(token_dir) if f.endswith('.tokens')]
    
    def __iter__(self):
        for file in tqdm(self.files, desc="Reading token files", total=len(self.files)):
            try:
                with open(file, 'r', encoding='utf-8') as f:
                    tokens = f.read().split()
                    yield tokens
            except Exception as e:
                print(f"Error reading {file}: {e}")
                continue

In [3]:
pretrained_model = KeyedVectors.load_word2vec_format(glove_path, binary=False)

In [4]:
corpus = TokenCorpus(TOKEN_DIR)

model = Word2Vec(vector_size=50, window=8, sg=1, hs=0, negative=10, workers=6)


In [5]:
model.build_vocab(corpus)

Reading token files: 100%|██████████| 81433/81433 [02:39<00:00, 510.93it/s]


In [6]:
# 1. Identify words that exist in both models using set intersection (Instant)
common_words = list(set(model.wv.index_to_key) & set(pretrained_model.key_to_index))

if common_words:
    pretrained_vectors = pretrained_model[common_words]
    model.wv.add_vectors(common_words, pretrained_vectors, replace=True)

print(f"Injected {len(common_words)} vectors instantly.")

Injected 199107 vectors instantly.


In [7]:
model.train(corpus, total_examples=model.corpus_count, epochs=5)

Reading token files: 100%|██████████| 81433/81433 [21:41<00:00, 62.57it/s]  


(1055689403, 1404768550)

In [9]:
# Testing the model
model.wv.similarity("king", "queen")

0.7557059

In [10]:
model.wv.similarity("king", "banana")


0.17256366

In [11]:
model.wv.similarity("asim", "munir")


0.83946186

In [18]:
model.wv.similarity("terrorist", "israel")


0.6234075

In [43]:
model.wv.save_word2vec_format('fine_tunned_model.word2vec.txt')

In [44]:
model.train(corpus, total_examples=model.corpus_count, epochs=3, compute_loss=True)

Reading token files: 100%|██████████| 81433/81433 [23:10<00:00, 58.58it/s] 


(633414081, 842861130)

In [60]:
model.wv.save_word2vec_format('fine_tunned_model.word2vec.txt')